# Load COCO Reference Images and Libraries

In [ ]:
#Author: Jaden Barnwell, 2026
!pip install -q --upgrade huggingface_hub accelerate datasets torch-fidelity torchmetrics pillow tqdm requests
!pip show torch transformers accelerate
!pip install diffusers==0.20.0 transformers==4.35.0 -q
#!pip install diffusers==0.14.0 transformers==4.30.0 -q
!pip install opencv-python-headless -q
!pip install controlnet-aux -q
!pip install ptflops -q

In [2]:
import os
import sys

if not os.path.exists('Faster-Diffusion'):
    os.system('git clone https://github.com/hutaiHang/Faster-Diffusion.git')
else:
    print('Faster-Diffusion repo already cloned')

sys.path.insert(0, './Faster-Diffusion')

Faster-Diffusion repo already cloned


In [ ]:
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from torch_fidelity import calculate_metrics
from utils_sd import register_parallel_pipeline, register_faster_forward
from PIL import Image
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel
from ptflops import get_model_complexity_info

In [4]:
!pip freeze > requirements.txt

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

BATCH_SIZE = 32
NUM_STEPS = 25
NUM_IMAGES = 1000

OUTPUT_ROOT = "/workspace/outputs"
COCO_DIR = "/workspace/coco_real"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(COCO_DIR, exist_ok=True)

print(f"Device: {DEVICE}, Batch: {BATCH_SIZE}")

Device: cuda, Batch: 32


In [6]:
import zipfile
import os

if not os.path.exists('val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip
    with zipfile.ZipFile('val2017.zip', 'r') as z:
        z.extractall('.')

if not os.path.exists('annotations'):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    with zipfile.ZipFile('annotations_trainval2017.zip', 'r') as z:
        z.extractall('.')

print("COCO ready")

COCO ready


In [7]:
import zipfile

!rm -rf val2017
with zipfile.ZipFile('val2017.zip', 'r') as z:
    z.extractall('.')
print("Re-extracted val2017")

Re-extracted val2017


In [8]:
COCO_IMG_DIR = "val2017"
COCO_ANN_FILE = "annotations/captions_val2017.json"


def load_coco_val2017():
    with open(COCO_ANN_FILE, "r") as f:
        data = json.load(f)

    # Map image_id -> captions
    captions_dict = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        captions_dict.setdefault(img_id, []).append(ann["caption"])

    # Map image_id -> filename
    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}

    samples = []
    for img_id, file_name in id_to_file.items():
        path = os.path.join(COCO_IMG_DIR, file_name)
        captions = captions_dict.get(img_id, [])

        if os.path.exists(path) and captions:
            samples.append({
                "image_path": path,
                "caption": captions[0]  # pick first caption
            })

    return samples


samples = load_coco_val2017()
print(f"Loaded {len(samples)} samples")

Loaded 5000 samples


In [9]:
import os
from PIL import Image

OUTPUT_REAL = "/workspace/coco_real"
os.makedirs(OUTPUT_REAL, exist_ok=True)

saved = 0
skipped = 0

for i, sample in enumerate(samples):
    try:
        img = Image.open(sample["image_path"]).convert("RGB")
        img = img.resize((512, 512))
        img.save(f"{OUTPUT_REAL}/{i}.png")
        saved += 1
    except Exception as e:
        print(f"Skipping {sample['image_path']}: {e}")
        skipped += 1

print(f"Saved: {saved}, Skipped: {skipped}")

Saved: 5000, Skipped: 0


In [10]:
!df -h
!pwd
!ls /workspace/

Filesystem                   Size  Used Avail Use% Mounted on
overlay                       40G  1.1G   39G   3% /
tmpfs                         64M     0   64M   0% /dev
mfs#us-wa-1.runpod.net:9421  479T  245T  234T  52% /workspace
shm                          117G     0  117G   0% /dev/shm
/dev/md127                    28T  4.7T   24T  17% /etc/hosts
/dev/nvme1n1p2               1.8T   33G  1.6T   2% /usr/bin/nvidia-smi
tmpfs                       1002G     0 1002G   0% /sys/fs/cgroup
tmpfs                       1002G   12K 1002G   1% /proc/driver/nvidia
tmpfs                       1002G  4.0K 1002G   1% /etc/nvidia/nvidia-application-profiles-rc.d
tmpfs                        201G   64M  201G   1% /run/nvidia-persistenced/socket
tmpfs                       1002G     0 1002G   0% /proc/acpi
tmpfs                       1002G     0 1002G   0% /proc/scsi
tmpfs                       1002G     0 1002G   0% /sys/firmware
tmpfs                       1002G     0 1002G   0% /sys/devices/virtu

### Faster-Diffusion Computation w/ FID Metric

In [11]:
def load_prompts(samples, num_images):
    return [s["caption"] for s in samples[:num_images]]


prompts = load_prompts(samples, NUM_IMAGES)

def load_pipeline(name):
    if name == "sd1.5":
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=DTYPE
        )
    elif name == "sd2.1":
        pipe = DiffusionPipeline.from_pretrained(
            "sd2-community/stable-diffusion-2-1",
            torch_dtype=DTYPE
        )

    pipe = pipe.to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def enable_faster_diffusion(pipe, use_faster):
    if not use_faster:
        return
    register_parallel_pipeline(pipe, mod='50ls')
    pipe.unet.order = 0  # manually initialize order attribute
    register_faster_forward(pipe.unet, mod='50ls')

In [12]:
def generate_images(pipe, prompts, out_dir, use_faster):
    os.makedirs(out_dir, exist_ok=True)

    enable_faster_diffusion(pipe, use_faster)

    all_times = []

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        batch_size = len(batch_prompts)

        # SAME LATENTS for fairness
        latents = torch.randn(
            (batch_size, pipe.unet.in_channels, 64, 64),
            device=DEVICE,
            dtype=DTYPE
        )

        start = time.time()

        images = pipe(
            batch_prompts,
            num_inference_steps=NUM_STEPS,
            latents=latents
        ).images

        end = time.time()
        all_times.append(end - start)

        for j, img in enumerate(images):
            img.save(f"{out_dir}/{i+j}.png")

    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0

    return np.mean(all_times), np.std(all_times), peak_mem

def compute_fid(real_dir, fake_dir):
    metrics = calculate_metrics(
        input1=real_dir,
        input2=fake_dir,
        fid=True,
        isc=False,
        kid=False,
        cuda=(DEVICE == "cuda")
    )
    return metrics["frechet_inception_distance"]

In [13]:
NUM_IMAGES = 1000
prompts = load_prompts(samples, NUM_IMAGES)
print(f"Prompts: {len(prompts)}, Batches: {len(prompts) // BATCH_SIZE}")

Prompts: 1000, Batches: 31


In [ ]:
configs = [
    ("sd1.5", False),
    ("sd1.5", True),
    ("sd2.1", False),
    ("sd2.1", True),
]

results = {}

for model, faster in configs:
    print(f"\nRunning {model} | faster={faster}")

    pipe = load_pipeline(model)

    # Compute MACs for this config's UNet
    try:
        macs, _ = get_model_complexity_info(pipe.unet, (4, 64, 64), as_strings=False)
        macs_g = macs / 1e9
    except Exception as e:
        print(f"  MACs computation failed: {e}")
        macs_g = float('nan')

    out_dir = f"{OUTPUT_ROOT}/{model}_{'faster' if faster else 'base'}"

    mean_t, std_t, peak_mem = generate_images(pipe, prompts, out_dir, faster)

    time_mean_img = mean_t / BATCH_SIZE
    time_var_img = (std_t / BATCH_SIZE) ** 2
    time_std_img = std_t / BATCH_SIZE
    throughput = BATCH_SIZE / mean_t

    results[(model, faster)] = {
        "dir": out_dir,
        "time_mean_img": time_mean_img,
        "time_var_img": time_var_img,
        "time_std_img": time_std_img,
        "peak_vram_gb": peak_mem,
        "throughput_img_per_sec": throughput,
        "macs_g": macs_g,
    }

    del pipe
    torch.cuda.empty_cache()

In [ ]:
fid_results = {}

for model, faster in configs:
    out_dir = results[(model, faster)]["dir"]
    fid_val = compute_fid(COCO_DIR, out_dir)
    fid_results[(model, faster)] = fid_val

print("\n=== FID RESULTS ===")
for (model, faster), fid in fid_results.items():
    print(f"  {model} faster={faster}: FID = {fid:.3f}")

### CLIP Computation

In [16]:
def load_clip():
    model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32"
    ).to(DEVICE)

    processor = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32"
    )

    model.eval()
    return model, processor

In [17]:
def compute_clip_score(image_dir, prompts, model, processor):
    image_files = sorted(os.listdir(image_dir))

    scores = []

    for i in tqdm(range(0, len(image_files), BATCH_SIZE)):
        batch_files = image_files[i:i+BATCH_SIZE]
        batch_images = []
        batch_texts = []

        for j, file in enumerate(batch_files):
            img = Image.open(os.path.join(image_dir, file)).convert("RGB")
            batch_images.append(img)

            # match prompt index
            idx = i + j
            batch_texts.append(prompts[idx])

        inputs = processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            # Normalize (CRITICAL)
            image_embeds = F.normalize(image_embeds, dim=-1)
            text_embeds = F.normalize(text_embeds, dim=-1)

            # cosine similarity
            batch_scores = (image_embeds * text_embeds).sum(dim=-1)

        scores.extend(batch_scores.cpu().numpy())

    return float(np.mean(scores))

In [ ]:
clip_model, clip_processor = load_clip()

clip_results = {}

for model, faster in configs:
    out_dir = results[(model, faster)]["dir"]

    print(f"\nComputing CLIP for {model} faster={faster}...")

    clip_score = compute_clip_score(
        out_dir, prompts, clip_model, clip_processor
    )

    clip_results[(model, faster)] = clip_score

# Results

In [ ]:
col_keys = [
    ("sd1.5", False),
    ("sd1.5", True),
    ("sd2.1", False),
    ("sd2.1", True),
]
col_labels = ["SD1.5 False", "SD1.5 True", "SD2.1 False", "SD2.1 True"]
w = 14

header = f"{'Metric':<28}" + "".join(f"{lbl:>{w}}" for lbl in col_labels)
sep = "-" * len(header)

def _row(label, vals_fn):
    vals = "".join(f"{vals_fn(k):>{w}}" for k in col_keys)
    return f"{label:<28}" + vals

print(sep)
print(header)
print(sep)
print(_row("Avg Time Per Img (s)",  lambda k: f"{results[k]['time_mean_img']:.4f}"))
print(_row("Var Time Per Img (s)",  lambda k: f"{results[k]['time_var_img']:.6f}"))
print(_row("Std Time Per Img (s)",  lambda k: f"{results[k]['time_std_img']:.4f}"))
print(_row("VRAM (GB)",             lambda k: f"{results[k]['peak_vram_gb']:.2f}"))
print(_row("FID",                   lambda k: f"{fid_results[k]:.3f}"))
print(_row("CLIP",                  lambda k: f"{clip_results[k]:.4f}"))
print(_row("Throughput (img/s)",    lambda k: f"{results[k]['throughput_img_per_sec']:.3f}"))
print(_row("MACs (G)",              lambda k: f"{results[k]['macs_g']:.2f}"))
print(sep)